# Inference pipeline for the pre-trained models for [Iqra'Eval Challenge](https://huggingface.co/spaces/IqraEval/SharedTask_ArabicNLP2025)

## Installations

In [1]:
!pip uninstall torchaudio torch

Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchaudio-2.10.0+cu128.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchaudio/*
Proceed (Y/n)? Y
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Would remove:
    /usr/local/bin/torchfrtrace
    /usr/local/bin/torchrun
    /usr/local/lib/python3.12/dist-packages/functorch/*
    /usr/local/lib/python3.12/dist-packages/torch-2.10.0+cu128.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torch/*
    /usr/local/lib/python3.12/dist-packages/torchgen/*
Proceed (Y/n)? Y
  Successfully uninstalled torch-2.10.0+cu128


In [ ]:
%%capture

!git clone https://github.com/s3prl/s3prl.git \
!cd s3prl \
!pip install -e ".[all]"
!pip install tensorboardX
!git clone https://github.com/Iqra-Eval/interspeech_IqraEval.git
!pip install torchcodec
!pip install -U datasets
!pip install jiwer


In [ ]:
# # !pip uninstall torch torchaudio
# import torchaudio
# import types

# # Patch removed API: set_audio_backend
# if not hasattr(torchaudio, "set_audio_backend"):
#     torchaudio.set_audio_backend = lambda *args, **kwargs: None

# # Patch removed API: torchaudio.sox_effects
# if not hasattr(torchaudio, "sox_effects"):
#     torchaudio.sox_effects = types.SimpleNamespace()

# if not hasattr(torchaudio.sox_effects, "apply_effects_tensor"):
#     def _no_sox_effects(wav, sr, effects=None):
#         # passthrough: return input unchanged
#         return wav, sr

#     torchaudio.sox_effects.apply_effects_tensor = _no_sox_effects


## Imports, restart session if you see any errros due to imports

In [ ]:
!pip install torch==2.1.0+cu121 torchaudio==2.1.0+cu121 torchvision==0.16.0+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torch==2.1.0+cu121 (from versions: 2.2.0+cu121, 2.2.1+cu121, 2.2.2+cu121, 2.3.0+cu121, 2.3.1+cu121, 2.4.0+cu121, 2.4.1+cu121, 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.1.0+cu121


In [ ]:
# Import Libraries
import torch
import torchaudio
import s3prl
import warnings
import argparse
from pathlib import Path
import tempfile
import requests
import os
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'torch'

In [ ]:
#@title S3PRLModel utility code (click to expand)
def download_if_needed(path):
    """Download file if path is a URL, else return path unchanged."""
    if str(path).startswith("http://") or str(path).startswith("https://"):
        response = requests.get(path)
        response.raise_for_status()
        suffix = os.path.splitext(str(path))[-1]
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        tmp.write(response.content)
        tmp.close()
        return tmp.name
    return path

class S3PRLModel:
    def __init__(self, ckpt, dict_path='dict.txt'):
        from s3prl.downstream.runner import Runner

        # Support local or remote paths for ckpt and dict
        ckpt = download_if_needed(ckpt)
        dict_path = download_if_needed(dict_path)
        torch.serialization.add_safe_globals([argparse.Namespace])
        model_dict = torch.load(ckpt, map_location='cpu', weights_only=False)
        self.args = model_dict['Args']
        self.config = model_dict['Config']

        # Config patch
        self.args.init_ckpt = ckpt
        self.args.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.config['downstream_expert']['text']["vocab_file"] = dict_path
        self.config['runner']['upstream_finetune'] = False
        self.config['runner']['layer_drop'] = False
        self.config['runner']['downstream_pretrained'] = None

        runner = Runner(self.args, self.config)
        self.upstream = runner._get_upstream()
        self.featurizer = runner._get_featurizer()
        self.downstream = runner._get_downstream()
        # For temp file cleanup
        self._temp_ckpt = ckpt if ckpt.startswith(tempfile.gettempdir()) else None
        self._temp_dict = dict_path if dict_path.startswith(tempfile.gettempdir()) else None

    def __call__(self, wav_path):
        wav, sr = torchaudio.load(wav_path)
        wav = wav.mean(0).unsqueeze(0)  # Convert to mono
        wav = wav.to(self.args.device)

        # Prepare dummy inputs
        dummy_split = "inference"
        dummy_filenames = [Path(wav_path).stem]  # Use filename as ID
        dummy_records = {"loss": [], "hypothesis": [], "groundtruth": [], "filename": []}

        with torch.no_grad():
            features = self.upstream.model(wav)
            features = self.featurizer.model(wav, features)
            dummy_labels = [[] for _ in features]  # Empty labels
            self.downstream.model(dummy_split, features, dummy_labels, dummy_filenames, dummy_records)
            predictions = dummy_records["hypothesis"]

        return predictions

    def cleanup(self):
        # Clean up downloaded temporary files
        for f in [self._temp_ckpt, self._temp_dict]:
            if f and os.path.isfile(f):
                os.remove(f)

def process_directory(ckpt, dict_path, wav_dir, output_csv):
    model = S3PRLModel(ckpt, dict_path)
    wav_files = list(Path(wav_dir).glob("*.wav"))
    wav_files.sort()
    results = []

    for wav_file in wav_files:
        output = model(str(wav_file))[0]
        print(f"{wav_file.name}: {output}")
        results.append({"ID": wav_file.name.split('.')[0], "Prediction": output})

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Results saved to {output_csv}")
    # Clean up temp files if any
    model.cleanup()



## Inference using pre-trained HuBERT_BASE phoneme recognition model on a subset available as [sample example](https://huggingface.co/datasets/IqraEval/dummy_samples)

In [ ]:
# !pip install torchcodec

In [ ]:
#@title  Download sample dataset and save groundtruth: Reference_phoneme and Annotation_phoneme
from datasets import load_dataset
import os
from scipy.io import wavfile
import pandas as pd


# Download dataset
dataset = load_dataset("IqraEval/dummy_samples", split="train")

# Directory for wav files
wav_dir = "/content/dummy_wavs"
os.makedirs(wav_dir, exist_ok=True)

# Prepare test.csv and save wav files
test_rows = []
for i, sample in enumerate(dataset):
    uid = sample['ID']
    ref_phn = sample['Reference_phoneme']
    ann_phn = sample['Annotation_phoneme']
    test_rows.append({
        "ID": uid,
        "Reference_phn": ref_phn,
        "Annotation_phn": ann_phn
    })
    wav_path = os.path.join(wav_dir, f"{uid}.wav")
    wavfile.write(wav_path, sample['audio']['sampling_rate'], sample['audio']['array'])
print(f"Saved {len(dataset)} wav files to {wav_dir}")

# Save test.csv in Colab root for the evaluation scripts
test_df = pd.DataFrame(test_rows)
test_df.to_csv("/content/test.csv", index=False)
print("Prepared test.csv")

Saved 4 wav files to /content/dummy_wavs
Prepared test.csv


## Model prediction

In [ ]:

%%capture
# ============= USER VARIABLES (EDIT THESE) =============
# Provide local path or URL to checkpoint
ckpt = "https://huggingface.co/IqraEval/Iqra_hubert_base/resolve/main/hubert_base.ckpt" # specify model, for e.g., wavlm_base.ckpt
dict_path = "/content/interspeech_IqraEval/vocab/sws_arabic.txt"   # specify vocab
wav_dir = "/content/dummy_wavs"   # e.g., "/content/wavs"
output_csv = "/content/results.csv" # e.g., "/content/results.csv"
# ============= END USER VARIABLES ======================

# # To upload files from your local machine to Colab, uncomment and use:
# from google.colab import files
# uploaded = files.upload()  # Will prompt to upload files

# ============= RUN PROCESS ============================
process_directory(ckpt, dict_path, wav_dir, output_csv)


ModuleNotFoundError: No module named 'torchaudio.sox_effects'

In [ ]:
print("torchaudio version:", torchaudio.__version__)

torchaudio version: 2.9.0+cu126


In [ ]:
df_result = pd.read_csv("/content/results.csv")
df_test = pd.read_csv("/content/test.csv")
display(df_result)
display(df_test)

,ID,Prediction
0,00000_00003,w a < i * b t a l aa E i b r aa h ii m u r a b...
1,00000_00004,m aa n a n s a x U m i n < aa y a
2,00000_00005,y a x A $ ii k u m nn u E aa s u < a m a n a m...
3,00000_00006,w a l i y a s t a E f i ll a * ii n a l aa y a...


,ID,Reference_phn,Annotation_phn
0,00000_00003,w a < i * i b t a l aa < i b r aa h ii m a r a...,w a < i * i b t a l aa < i b r aa h ii m u r a...
1,00000_00004,m aa n a n s a x m i n < aa y a t i n,m aa n a n s a x U m i n < aa y a t i n
2,00000_00005,y u g $ ii k u m u nn u E aa s a < a m a n a t...,y u x $ ii k u m u nn u E < a u < a m a n a t ...
3,00000_00006,w a l y a s t a E f i f ll a * ii n a l aa y a...,w a l y a s t a E f i ll a * ii n a l aa y a j...


In [ ]:
%cd /content/interspeech_IqraEval/mdd_eval/

#Align Data
!python align_data.py --truth-file /content/test.csv --pred-file /content/results.csv

/content/interspeech_IqraEval/mdd_eval
** MD&D Evaluation **
Correct Rate: 0.844
Accuracy: 0.8257


In [ ]:
#@title  Metrics Calculation (click to expand)
import subprocess
import pandas as pd
import re

# Run the script and capture output
result = subprocess.run(
    ["python", "ins_del_cor_sub_analysis.py", "--align-folder", "aligned/"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

output = result.stdout

# Parse the metrics from the output
pattern = re.compile(r"([A-Za-z ]+):\s*([0-9.]+)(?:\s*\|\s*([0-9]+))?")
rows = []
for line in output.splitlines():
    m = pattern.match(line.strip())
    if m:
        metric, value, count = m.groups()
        rows.append({
            "Metric": metric,
            "Value": float(value),
            "Count": int(count) if count else None
        })

# Display as table
df = pd.DataFrame(rows)
display(df)
# !python ins_del_cor_sub_analysis.py --align-folder aligned/

,Metric,Value,Count
0,Recall,0.8571,NaN
1,Precision,0.2727,NaN
2,True Acception,0.8462,88.0
3,False Rejection,0.1538,16.0
4,False Acceptance,0.1429,1.0
5,Correct Diagnosis,0.8333,5.0
6,Error Diagnosis,0.1667,1.0
7,False Acceptance Rate,0.1429,NaN
8,False Rejection Rate,0.1538,NaN
9,Diagnosis Error Rate,0.1667,NaN


In [ ]:
#@title  PER Calculation (click to expand)
from jiwer import wer
# Load files
results = pd.read_csv('/content/results.csv')
truth = pd.read_csv('/content/test.csv')

# Merge on ID
merged = pd.merge(results, truth, on='ID')

def compute_per(truth_col):
    ref_corpus = " ".join(merged[truth_col].astype(str).tolist())
    hyp_corpus = " ".join(merged["Prediction"].astype(str).tolist())
    return wer(ref_corpus, hyp_corpus)

per_ref = compute_per("Reference_phn")
per_ann = compute_per("Annotation_phn")

# Display as a table
df = pd.DataFrame({
    "Reference": ["Reference_phn", "Annotation_phn"],
    "PER": [per_ref, per_ann]
})
display(df)

,Reference,PER
0,Reference_phn,0.201835
1,Annotation_phn,0.174312
